# 01_Voice Preprocessing

## Objective
The objective of this notebook is to preprocess voice recordings and standardize audio inputs.

## Method
Audio preprocessing and normalization pipeline.

## Input
Raw voice recordings.

## Processing Tasks

1. Voice Dataset Inspection.
2. Audio Dataset Statistics.
3. Voice Audio Renaming.
4. Audio Loading & Inspection.
5. Audio Standardization.
6. Standardization Validation.
7. Audio Signal Analysis.
8. Waveform Visualization.
9. Complete Audio Signal Analysis.

## Output
- Standardized voice audio files.
- Preprocessing quality visual checks.
- Prepared audio for feature extraction.

## Purpose
Create consistent voice inputs for robust feature extraction and modeling.


In [ ]:

from pathlib import Path

import numpy as np
import pandas as pd
import soundfile as sf
import matplotlib.pyplot as plt

PROJECT_PATH = Path(r"E:\Multimodal Biometric Verification System")

RAW_VOICE_PATH = (
    PROJECT_PATH / "Dataset" / "Raw" / "Voice"
)

PROCESSED_VOICE_PATH = (
    PROJECT_PATH / "Dataset" / "Processed" / "Voice"
)

USERS = ["User_01", "User_02"]

AUDIO_EXTENSIONS = [
    ".wav",
    ".mp3",
    ".flac",
    ".ogg",
    ".m4a"
]

print("Environment configured successfully.")
print(f"Raw Voice Path: {RAW_VOICE_PATH}")
print(f"Processed Voice Path: {PROCESSED_VOICE_PATH}")



# 1. Voice Dataset Inspection

This section examines the raw voice recordings for each user.

The inspection includes:
- Number of audio recordings.
- Audio file names and formats.
- Sample rates.
- Number of channels.
- Recording durations.
- Total number of recordings per user.


In [ ]:

audio_records = []

for user in USERS:

    user_path = RAW_VOICE_PATH / user

    audio_files = [
        file for file in user_path.iterdir()
        if file.is_file() and file.suffix.lower() in AUDIO_EXTENSIONS
    ]

    print(f"{user}: {len(audio_files)} recordings found.")

    for audio_file in audio_files:

        try:
            audio_info = sf.info(audio_file)

            audio_records.append({
                "User_ID": user,
                "Audio_Name": audio_file.name,
                "Format": audio_info.format,
                "Sample_Rate": audio_info.samplerate,
                "Channels": audio_info.channels,
                "Duration": audio_info.duration,
                "Frames": audio_info.frames
            })

        except Exception as error:
            print(f"Error reading {audio_file.name}: {error}")

voice_dataset = pd.DataFrame(audio_records)

print("\nTotal Audio Recordings:", len(voice_dataset))

display(voice_dataset.head(10))



# 2. Audio Dataset Statistics


In [ ]:

print("Recording Distribution:")
display(voice_dataset["User_ID"].value_counts())

print("\nSample Rate Distribution:")
display(voice_dataset["Sample_Rate"].value_counts())

print("\nChannel Distribution:")
display(voice_dataset["Channels"].value_counts())

print("\nDuration Statistics:")
display(
    voice_dataset.groupby("User_ID")["Duration"].describe()
)

print("\nOverall Duration Range:")
print(f"Minimum: {voice_dataset['Duration'].min():.2f} seconds")
print(f"Maximum: {voice_dataset['Duration'].max():.2f} seconds")

print("\nMissing Values:")
display(voice_dataset.isnull().sum())



# 3. Voice Audio Renaming


In [ ]:

for user in USERS:

    user_path = RAW_VOICE_PATH / user

    audio_files = sorted([
        file for file in user_path.iterdir()
        if file.is_file() and file.suffix.lower() in AUDIO_EXTENSIONS
    ])

    print(f"\nRenaming recordings for {user}:")

    for index, old_file in enumerate(audio_files, start=1):

        new_name = f"{user}_{index:03d}{old_file.suffix.lower()}"
        new_file = user_path / new_name

        if old_file.name == new_name:
            continue

        if new_file.exists():
            raise FileExistsError(
                f"Target file already exists: {new_file}"
            )

        old_file.rename(new_file)

        print(f"{old_file.name} -> {new_name}")

    print(f"Total renamed: {len(audio_files)}")


# 4. Audio Loading & Inspection


In [ ]:
# Select one audio recording from each user

sample_audio_files = {}

for user in USERS:

    user_path = RAW_VOICE_PATH / user

    audio_files = sorted([
        file for file in user_path.iterdir()
        if file.is_file() and file.suffix.lower() in AUDIO_EXTENSIONS
    ])

    sample_audio_files[user] = audio_files[0]


# Load and inspect selected recordings

loaded_audio_samples = {}

for user, audio_file in sample_audio_files.items():

    audio_signal, sample_rate = sf.read(audio_file)

    loaded_audio_samples[user] = {
        "signal": audio_signal,
        "sample_rate": sample_rate,
        "file_name": audio_file.name
    }

    print(f"\n{user}")
    print("-" * 40)
    print(f"File Name: {audio_file.name}")
    print(f"Sample Rate: {sample_rate} Hz")
    print(f"Signal Shape: {audio_signal.shape}")
    print(f"Data Type: {audio_signal.dtype}")
    print(f"Duration: {len(audio_signal) / sample_rate:.2f} seconds")

    if audio_signal.ndim == 2:
        print(f"Channels: {audio_signal.shape[1]}")
    else:
        print("Channels: 1 (Mono)")


# Visualize audio waveforms

fig, axes = plt.subplots(2, 1, figsize=(14, 7))

for ax, (user, audio_data) in zip(axes, loaded_audio_samples.items()):

    signal = audio_data["signal"]
    sample_rate = audio_data["sample_rate"]

    time_axis = np.arange(len(signal)) / sample_rate

    if signal.ndim == 2:
        ax.plot(time_axis, signal[:, 0], label="Left Channel")
        ax.plot(time_axis, signal[:, 1], alpha=0.7, label="Right Channel")
        ax.legend()
    else:
        ax.plot(time_axis, signal)

    ax.set_title(f"{user} - Audio Waveform")
    ax.set_xlabel("Time (seconds)")
    ax.set_ylabel("Amplitude")
    ax.grid(True)

plt.tight_layout()
plt.show()

# 5. Audio Standardization


In [ ]:
import os

STANDARDIZED_PATH = (
    PROCESSED_VOICE_PATH / "Audio_Standardized"
)

TARGET_SAMPLE_RATE = 16000

for user in USERS:
    (STANDARDIZED_PATH / user).mkdir(
        parents=True,
        exist_ok=True
    )

print("Audio Standardization directories created successfully.")

for user in USERS:
    print(f"{user}: {STANDARDIZED_PATH / user}")

In [ ]:
import soundfile as sf
from scipy.signal import resample_poly
from math import gcd

standardization_records = []

for user in USERS:

    user_path = RAW_VOICE_PATH / user
    output_user_path = STANDARDIZED_PATH / user

    audio_files = sorted([
        file for file in user_path.iterdir()
        if file.is_file() and file.suffix.lower() in AUDIO_EXTENSIONS
    ])

    for audio_file in audio_files:

        # Load audio
        audio_signal, original_sample_rate = sf.read(audio_file)

        # Convert Stereo to Mono
        if audio_signal.ndim == 2:
            audio_signal = np.mean(audio_signal, axis=1)

        # Resample to 16 kHz
        divisor = gcd(original_sample_rate, TARGET_SAMPLE_RATE)

        audio_signal = resample_poly(
            audio_signal,
            TARGET_SAMPLE_RATE // divisor,
            original_sample_rate // divisor
        )

        # Save as WAV using 16-bit PCM
        output_file = output_user_path / f"{audio_file.stem}.wav"

        sf.write(
            output_file,
            audio_signal,
            TARGET_SAMPLE_RATE,
            subtype="PCM_16"
        )

        standardization_records.append({
            "User_ID": user,
            "Original_Name": audio_file.name,
            "Standardized_Name": output_file.name,
            "Original_Sample_Rate": original_sample_rate,
            "Standardized_Sample_Rate": TARGET_SAMPLE_RATE,
            "Duration": len(audio_signal) / TARGET_SAMPLE_RATE
        })

        print(f"{user}: {audio_file.name} -> {output_file.name}")

standardization_df = pd.DataFrame(standardization_records)

print("\nAudio Standardization Completed.")
print(f"Total Processed Recordings: {len(standardization_df)}")

# 6. Standardization Validation


In [ ]:
validation_records = []

for user in USERS:

    user_path = STANDARDIZED_PATH / user

    audio_files = sorted(user_path.glob("*.wav"))

    for audio_file in audio_files:

        audio_info = sf.info(audio_file)

        validation_records.append({
            "User_ID": user,
            "Audio_Name": audio_file.name,
            "Sample_Rate": audio_info.samplerate,
            "Channels": audio_info.channels,
            "Subtype": audio_info.subtype,
            "Duration": audio_info.duration
        })

validation_df = pd.DataFrame(validation_records)

print("Standardization Validation Results")
print("-" * 45)

print("\nRecording Distribution:")
print(validation_df["User_ID"].value_counts().sort_index())

print("\nSample Rate Distribution:")
print(validation_df["Sample_Rate"].value_counts())

print("\nChannel Distribution:")
print(validation_df["Channels"].value_counts())

print("\nAudio Subtype Distribution:")
print(validation_df["Subtype"].value_counts())

print("\nTotal Validated Recordings:", len(validation_df))

print(
    "\nAll Requirements Satisfied:",
    (
        len(validation_df) == 42
        and (validation_df["Sample_Rate"] == 16000).all()
        and (validation_df["Channels"] == 1).all()
        and (validation_df["Subtype"] == "PCM_16").all()
    )
)

# 7. Audio Signal Analysis


In [ ]:
signal_analysis_records = []

for user in USERS:

    user_path = STANDARDIZED_PATH / user

    audio_files = sorted(user_path.glob("*.wav"))

    for audio_file in audio_files:

        signal, sample_rate = sf.read(audio_file)

        peak_amplitude = np.max(np.abs(signal))

        rms_amplitude = np.sqrt(np.mean(signal ** 2))

        clipping_samples = np.sum(np.abs(signal) >= 0.999)

        signal_analysis_records.append({
            "User_ID": user,
            "Audio_Name": audio_file.name,
            "Peak_Amplitude": peak_amplitude,
            "RMS_Amplitude": rms_amplitude,
            "Clipping_Samples": clipping_samples
        })

signal_analysis_df = pd.DataFrame(signal_analysis_records)

print("Audio Signal Analysis Summary")
print("-" * 45)

display(
    signal_analysis_df.groupby("User_ID")[
        ["Peak_Amplitude", "RMS_Amplitude", "Clipping_Samples"]
    ].agg(["mean", "min", "max"])
)

print("\nTotal Clipping Samples:",
      signal_analysis_df["Clipping_Samples"].sum())



# 8. Waveform Visualization


In [ ]:
fig, axes = plt.subplots(2, 1, figsize=(14, 8))

waveform_records = []

for index, user in enumerate(USERS):

    audio_file = STANDARDIZED_PATH / user / f"{user}_001.wav"

    signal, sample_rate = sf.read(audio_file)

    time_axis = np.arange(len(signal)) / sample_rate

    # Plot Waveform
    axes[index].plot(time_axis, signal)

    axes[index].set_title(f"{user} - Waveform")
    axes[index].set_xlabel("Time (seconds)")
    axes[index].set_ylabel("Amplitude")
    axes[index].grid(True)

    # Signal Analysis
    peak_amplitude = np.max(np.abs(signal))
    rms_amplitude = np.sqrt(np.mean(signal ** 2))

    silence_threshold = 0.01
    silence_samples = np.sum(np.abs(signal) < silence_threshold)
    silence_percentage = (silence_samples / len(signal)) * 100

    waveform_records.append({
        "User_ID": user,
        "Duration": len(signal) / sample_rate,
        "Peak_Amplitude": peak_amplitude,
        "RMS_Amplitude": rms_amplitude,
        "Silence_Percentage": silence_percentage,
        "Minimum_Amplitude": np.min(signal),
        "Maximum_Amplitude": np.max(signal)
    })

plt.tight_layout()
plt.show()

# Display Text Analysis
waveform_df = pd.DataFrame(waveform_records)

print("\nWaveform Analysis Results")
print("-" * 60)

for _, record in waveform_df.iterrows():

    print(f"\n{record['User_ID']}")
    print(f"Duration: {record['Duration']:.2f} seconds")
    print(f"Peak Amplitude: {record['Peak_Amplitude']:.4f}")
    print(f"RMS Amplitude: {record['RMS_Amplitude']:.4f}")
    print(f"Silence Percentage: {record['Silence_Percentage']:.2f}%")
    print(f"Minimum Amplitude: {record['Minimum_Amplitude']:.4f}")
    print(f"Maximum Amplitude: {record['Maximum_Amplitude']:.4f}")

# 9. Complete Audio Signal Analysis

This section performs a signal-level analysis of all standardized voice recordings.

The analysis calculates:
- Peak Amplitude.
- RMS Amplitude.
- Silence Percentage.
- Recording Duration.


In [ ]:
complete_signal_records = []

SILENCE_THRESHOLD = 0.01

for user in USERS:

    user_path = STANDARDIZED_PATH / user

    audio_files = sorted(user_path.glob("*.wav"))

    for audio_file in audio_files:

        signal, sample_rate = sf.read(audio_file)

        peak_amplitude = np.max(np.abs(signal))

        rms_amplitude = np.sqrt(np.mean(signal ** 2))

        silence_samples = np.sum(
            np.abs(signal) < SILENCE_THRESHOLD
        )

        silence_percentage = (
            silence_samples / len(signal)
        ) * 100

        duration = len(signal) / sample_rate

        complete_signal_records.append({
            "User_ID": user,
            "Audio_Name": audio_file.name,
            "Duration": duration,
            "Peak_Amplitude": peak_amplitude,
            "RMS_Amplitude": rms_amplitude,
            "Silence_Percentage": silence_percentage
        })

complete_signal_df = pd.DataFrame(complete_signal_records)

print("Complete Audio Signal Analysis")
print("=" * 60)

print("\nRecording Distribution:")
print(complete_signal_df["User_ID"].value_counts().sort_index())

print("\nSignal Statistics:")

display(
    complete_signal_df.groupby("User_ID")[
        [
            "Duration",
            "Peak_Amplitude",
            "RMS_Amplitude",
            "Silence_Percentage"
        ]
    ].agg(["mean", "min", "max", "std"]).round(4)
)

print("\nTotal Analyzed Recordings:", len(complete_signal_df))